# Análisis de absentismo y bienestar laboral - 01/06/2026

## 1. Pregunta de negocio

> ¿Qué variables personales y laborales están asociadas con absentismo elevado y cómo ajustar políticas internas para reducir las ausencias?

## 2. Punto de partida: conexión con `analisis_perfil`

El notebook `Analisis_perfil_25052026.ipynb` ya revisó variables personales y laborales como edad, educación, hijos, distancia al trabajo, antigüedad, IMC, hábitos sociales, carga media y `Hit_target`.

Ese análisis mostró que el absentismo no se explica bien mediante relaciones lineales simples con variables individuales. Por eso, aquí no retrabajamos el análisis de perfil: lo usamos como base para ampliar la lectura desde absentismo y bienestar.

La lógica será:

1. revisar asociaciones personales/laborales sin afirmar causalidad;
2. si no hay una variable individual fuerte, pasar a impacto operativo;
3. proponer políticas internas basadas en motivos, momentos, concentración y segmentos agregados.


## 3. Dataset limpio utilizado y alcance mínimo


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
from matplotlib.colors import LinearSegmentedColormap

pd.set_option('display.max_columns', 100)
sns.set_style('white')
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True

# Paleta alineada con la entrega anterior
COLOR_FONDO = '#eef2ee'
COLOR_VERDE_OSCURO = '#064b25'
COLOR_VERDE_MEDIO = '#7fbd83'
COLOR_VERDE_CLARO = '#dfeedd'
COLOR_TEAL = '#376f6f'
COLOR_OCRE = '#c78d52'
COLOR_ALERTA = '#b46b4f'
COLOR_TEXTO = '#263238'

COLOR_PRINCIPAL = COLOR_VERDE_OSCURO
COLOR_SECUNDARIO = COLOR_TEAL

DATA_FILE = '../../Data/clean/clean_data_01062026.csv'
df_RRHH = pd.read_csv(DATA_FILE)


In [ ]:
alcance = pd.DataFrame({
    'métrica': [
        'registros limpios',
        'IDs observados',
        'horas totales de absentismo',
        'registros con ausencia > 0 horas'
    ],
    'valor': [
        len(df_RRHH),
        df_RRHH['ID'].nunique(),
        df_RRHH['Absenteeism_hours'].sum(),
        (df_RRHH['Absenteeism_hours'] > 0).sum()
    ]
})

alcance


**Lectura:** el análisis trabaja con el clean consolidado del 01/06/2026. Las métricas describen la muestra disponible, no la plantilla completa de RapidExpress.


## 4. Variables personales y laborales consideradas

**Variables personales:** `Age`, `Education_name`, `Son`, `Pet`, `Social_drinker`, `Social_smoker`, `BMI_calculated`.

**Variables laborales:** `Service_time`, `Distance_Residence_Work`, `Transportation_expense`, `Work_load_Average_day`, `Hit_target`, `Disciplinary_failure`.

**Variables operativas:** `Reason_absence_name`, `Month_absence_name`, `Day_week_name`, `Seasons_name`.


## 5. Construcción de absentismo elevado

Para comparar segmentos usamos una vista agregada por `ID`. Definimos `absentismo_elevado` como IDs en el cuartil superior de horas acumuladas.

Este criterio no es un modelo predictivo ni una etiqueta individual. Es una regla exploratoria para comparar grupos de forma transparente.


In [ ]:
df_empleados = df_RRHH.groupby('ID', as_index=False).agg(
    registros=('ID', 'size'),
    horas_totales=('Absenteeism_hours', 'sum'),
    media_horas=('Absenteeism_hours', 'mean'),
    Age=('Age', 'first'),
    Education_name=('Education_name', 'first'),
    Son=('Son', 'first'),
    Pet=('Pet', 'first'),
    Social_drinker=('Social_drinker', 'first'),
    Social_smoker=('Social_smoker', 'first'),
    BMI_calculated=('BMI_calculated', 'first'),
    Service_time=('Service_time', 'first'),
    Distance_Residence_Work=('Distance_Residence_Work', 'first'),
    Transportation_expense=('Transportation_expense', 'first'),
    Work_load_Average_day=('Work_load_Average_day', 'median'),
    Hit_target=('Hit_target', 'median'),
    Disciplinary_failure=('Disciplinary_failure', 'max')
)

umbral_absentismo_elevado = df_empleados['horas_totales'].quantile(0.75)
df_empleados['absentismo_elevado'] = df_empleados['horas_totales'] >= umbral_absentismo_elevado

print('Umbral de absentismo elevado:', umbral_absentismo_elevado, 'horas')
print('IDs con absentismo elevado:', df_empleados['absentismo_elevado'].sum())
print('IDs observados:', df_empleados['ID'].nunique())

df_empleados.sort_values('horas_totales', ascending=False)


**Lectura:** el cuartil superior permite separar los IDs con más horas acumuladas para comparar segmentos. 
No significa que esos IDs sean “de riesgo”; solo indica mayor carga registrada dentro de la muestra.


**Verificación importante sobre `Disciplinary_failure`:** esta variable indica si hay una nota/fallo disciplinario. En el dataset, las filas con `Disciplinary_failure = 1` tienen `Absenteeism_hours = 0`. Por tanto, no se interpreta como horas de absentismo ni como causa directa de ausencia. Al agregar por `ID` usamos `max` solo para saber si ese ID tuvo alguna nota disciplinaria en la muestra.

In [ ]:
pd.crosstab(df_RRHH['Disciplinary_failure'], df_RRHH['Absenteeism_hours'])

## 6. Revisión de asociaciones con variables personales/laborales

Primero recuperamos el enfoque de `analisis_perfil`: comparar absentismo medio por variables categóricas y revisar correlaciones en variables numéricas reales.

Se muestran variables representativas del análisis de perfil/desempeño. El enfoque puede ampliarse al resto de variables personales y laborales definidas, pero el objetivo aquí es comprobar si alguna variable individual separa claramente el absentismo.


In [ ]:
def comparar_absentismo(df, variable):
    resultado = (
        df.groupby(variable)['horas_totales']
        .mean()
        .round(2)
        .reset_index()
        .sort_values(by='horas_totales', ascending=False)
    )

    conteo = df[variable].value_counts()

    print("\nRecuento por grupo:")
    print(conteo)

    print("\nAbsentismo medio por grupo:")
    print(resultado)

    plt.figure(figsize=(8, 5))
    plt.gcf().patch.set_alpha(0)
    ax = plt.gca()
    ax.set_facecolor('none')
    sns.barplot(data=resultado, x=variable, y='horas_totales', color=COLOR_TEAL)
    plt.title(f'Absentismo medio por {variable}')
    plt.ylabel('Horas promedio de absentismo')
    plt.xlabel(variable)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:
comparar_absentismo(df_empleados, 'Education_name')


In [ ]:
comparar_absentismo(df_empleados, 'Son')


In [ ]:
comparar_absentismo(df_empleados, 'Social_drinker')


**Lectura:** las comparaciones por grupos muestran diferencias, pero no bastan para afirmar causalidad. Algunos grupos son pequeños y pueden estar afectados por pocos IDs con muchas horas.


In [ ]:
variables_corr = [
    'Age', 'Service_time', 'Distance_Residence_Work', 'Transportation_expense',
    'BMI_calculated', 'Work_load_Average_day', 'Hit_target', 'horas_totales'
]

corr = df_empleados[variables_corr].corr(method='spearman')

labels_es = {
    'Age': 'Edad',
    'Service_time': 'Antigüedad',
    'Distance_Residence_Work': 'Distancia',
    'Transportation_expense': 'Gasto transp.',
    'BMI_calculated': 'IMC',
    'Work_load_Average_day': 'Carga media',
    'Hit_target': 'Desempeño',
    'horas_totales': 'Horas absent.'
}

corr = corr.rename(index=labels_es, columns=labels_es)

cmap_verde = LinearSegmentedColormap.from_list(
    'verde_blanco_verde', ['#0d2014', '#ffffff', '#1e7c4b']
)
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

plt.figure(figsize=(10, 8), facecolor='none')
plt.gcf().patch.set_alpha(0)
ax = sns.heatmap(
    corr, annot=True, fmt='.2f', cmap=cmap_verde, center=0,
    mask=mask, square=True, vmin=-1, vmax=1, linewidths=0.5
)
ax.set_facecolor('none')
plt.title('Correlaciones variables personales/laborales y absentismo')
plt.xticks(rotation=35, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
correlacion_absentismo = (
    corr[['Horas absent.']]
    .drop(index='Horas absent.')
    .sort_values('Horas absent.', key=abs, ascending=False)
)

correlacion_absentismo


In [ ]:
correlacion_plot = (
    correlacion_absentismo
    .reset_index()
    .rename(columns={'index': 'variable', 'Horas absent.': 'r_spearman'})
)

plt.figure(figsize=(8, 4.5))
plt.gcf().patch.set_alpha(0)
ax = plt.gca()
ax.set_facecolor('none')
colores = [COLOR_TEAL if valor >= 0 else COLOR_OCRE for valor in correlacion_plot['r_spearman']]

sns.barplot(
    data=correlacion_plot,
    y='variable',
    x='r_spearman',
    palette=colores,
    hue='variable',
    legend=False
)

plt.axvline(0, color='black', linewidth=0.8)
plt.xlim(-0.4, 0.4)
plt.title('Spearman: variables personales/laborales vs horas de absentismo')
plt.xlabel('r de Spearman')
plt.ylabel('')
plt.tight_layout()
plt.show()


**Lectura del gráfico Spearman:** las barras muestran la fuerza y dirección de la asociación entre cada variable y las horas acumuladas de absentismo. Todas quedan cerca de cero; por eso se interpretan como señales débiles, no como explicación fuerte.


**Lectura:** las correlaciones con absentismo son débiles. La señal más alta queda lejos de una relación fuerte, por lo que no es riguroso afirmar que una variable personal o laboral explique por sí sola las ausencias.


## 6.1. Contrastes estadísticos por variable

Para cerrar mejor la pregunta de negocio, además de correlaciones se aplican tests no paramétricos similares a los usados en `Analisis_perfil` por Noelia y Sabina.

La unidad de análisis en esta sección es `df_empleados`, es decir, una fila por `ID`. Esto evita que una persona con muchos registros pese más que otra solo por aparecer más veces en el dataset. Sobre esa vista agregada se estudia:

- **Spearman:** variables numéricas reales vs horas acumuladas.
- **Mann-Whitney U:** variables binarias vs horas acumuladas.
- **Kruskal-Wallis:** variables con 3 o más grupos vs horas acumuladas.
- **Chi-cuadrado:** relación entre categorías y `absentismo_elevado`.

Las variables categorizadas (`Tiene_hijos`, `Tiene_mascotas`, tramos de edad, antigüedad, distancia, transporte, hit target e IMC) son agrupaciones de estudio. No sustituyen las variables originales ni crean perfiles cerrados.

Estos tests no prueban causalidad. Sirven para comprobar si alguna variable personal/laboral separa claramente el absentismo antes de pasar a impacto operativo.


In [ ]:
# Categorías auxiliares para los tests. No sustituyen a los datos originales.
df_empleados_tests = df_empleados.copy()

df_empleados_tests['Tiene_hijos'] = (df_empleados_tests['Son'] > 0).astype(int)
df_empleados_tests['Tiene_mascotas'] = (df_empleados_tests['Pet'] > 0).astype(int)
df_empleados_tests['Educacion_binaria'] = np.where(
    df_empleados_tests['Education_name'] == 'Secundaria',
    'Secundaria',
    'Estudios superiores'
)

df_empleados_tests['tramo_edad'] = pd.cut(
    df_empleados_tests['Age'],
    bins=[0, 30, 40, 50, 100],
    labels=['<=30', '31-40', '41-50', '>50']
)

df_empleados_tests['tramo_antiguedad'] = pd.cut(
    df_empleados_tests['Service_time'],
    bins=[0, 5, 10, 15, 40],
    labels=['<=5 años', '6-10 años', '11-15 años', '>15 años']
)

df_empleados_tests['tramo_distancia'] = pd.cut(
    df_empleados_tests['Distance_Residence_Work'],
    bins=[0, 10, 20, 30, 100],
    labels=['<=10 km', '11-20 km', '21-30 km', '>30 km']
)

df_empleados_tests['tramo_transporte'] = pd.qcut(
    df_empleados_tests['Transportation_expense'],
    q=4,
    duplicates='drop',
    labels=['Q1 bajo', 'Q2 medio-bajo', 'Q3 medio-alto', 'Q4 alto']
)

df_empleados_tests['tramo_hit_target'] = pd.qcut(
    df_empleados_tests['Hit_target'],
    q=3,
    duplicates='drop',
    labels=['Bajo', 'Medio', 'Alto']
)

df_empleados_tests['tramo_imc'] = pd.cut(
    df_empleados_tests['BMI_calculated'],
    bins=[0, 24.9, 29.9, 100],
    labels=['Normal', 'Sobrepeso', 'Obesidad']
)


In [ ]:
# Spearman para variables numéricas reales
variables_numericas_tests = [
    'Age', 'Service_time', 'Distance_Residence_Work', 'Transportation_expense',
    'Work_load_Average_day', 'Hit_target', 'BMI_calculated', 'Son', 'Pet'
]

resultados_spearman = []
for variable in variables_numericas_tests:
    rho, p_value = stats.spearmanr(
        df_empleados_tests[variable],
        df_empleados_tests['horas_totales'],
        nan_policy='omit'
    )
    resultados_spearman.append({
        'variable': variable,
        'rho_spearman': round(rho, 4),
        'p_value': round(p_value, 4),
        'significativo_p_005': 'Sí' if p_value < 0.05 else 'No'
    })

spearman_tests = (
    pd.DataFrame(resultados_spearman)
    .sort_values('rho_spearman', key=lambda s: s.abs(), ascending=False)
)

spearman_tests


**Lectura Spearman:** la asociación más alta es `Transportation_expense` con una rho aproximada de 0,32. Es significativa, pero sigue siendo débil/moderada baja y no permite explicar el absentismo por sí sola. `Son` aparece con una señal débil; el resto de variables numéricas queda cerca de cero.

In [ ]:
def test_mann_whitney(df, variable, target='horas_totales'):
    grupos = []
    nombres = []

    for nombre, grupo in df.groupby(variable, observed=False):
        valores = grupo[target].dropna()
        if len(valores) > 0:
            grupos.append(valores)
            nombres.append(nombre)

    if len(grupos) != 2:
        return None

    stat, p_value = stats.mannwhitneyu(grupos[0], grupos[1], alternative='two-sided')

    return {
        'variable': variable,
        'test': 'Mann-Whitney U',
        'grupos': ' vs '.join(map(str, nombres)),
        'n_grupos': [len(g) for g in grupos],
        'medianas': [round(float(g.median()), 2) for g in grupos],
        'p_value': round(p_value, 4),
        'significativo_p_005': 'Sí' if p_value < 0.05 else 'No'
    }

variables_binarias_tests = [
    'Tiene_hijos', 'Tiene_mascotas', 'Educacion_binaria',
    'Social_drinker', 'Social_smoker', 'Disciplinary_failure'
]

mann_whitney_tests = pd.DataFrame([
    test_mann_whitney(df_empleados_tests, variable)
    for variable in variables_binarias_tests
])

mann_whitney_tests


**Lectura Mann-Whitney:** en las variables binarias no aparece una separación generalizada. `Disciplinary_failure` puede salir significativo en la vista agregada por ID, pero no debe leerse como horas de absentismo: en las filas donde vale 1, las horas son 0. Solo indica que algunos IDs con nota disciplinaria también acumulan absentismo en otros registros. `Social_smoker` aparece significativo, aunque el grupo es pequeño, por lo que no conviene convertirlo en conclusión fuerte.

In [ ]:
def test_kruskal(df, variable, target='horas_totales'):
    grupos = []
    nombres = []

    for nombre, grupo in df.groupby(variable, observed=False):
        valores = grupo[target].dropna()
        if len(valores) > 0:
            grupos.append(valores)
            nombres.append(nombre)

    if len(grupos) < 3:
        return None

    stat, p_value = stats.kruskal(*grupos)

    return {
        'variable': variable,
        'test': 'Kruskal-Wallis',
        'grupos': ', '.join(map(str, nombres)),
        'n_grupos': [len(g) for g in grupos],
        'medianas': [round(float(g.median()), 2) for g in grupos],
        'p_value': round(p_value, 4),
        'significativo_p_005': 'Sí' if p_value < 0.05 else 'No'
    }

variables_grupos_tests = [
    'tramo_edad', 'tramo_antiguedad', 'tramo_distancia',
    'tramo_transporte', 'tramo_hit_target', 'tramo_imc', 'Son', 'Pet'
]

kruskal_tests = pd.DataFrame([
    test_kruskal(df_empleados_tests, variable)
    for variable in variables_grupos_tests
])

kruskal_tests


**Lectura Kruskal-Wallis:** al agrupar variables continuas por tramos aparecen señales en `tramo_transporte`, `tramo_hit_target` y `tramo_antiguedad`. Esto indica que algunas diferencias se ven mejor por segmentos agregados que con una correlación lineal simple.

In [ ]:
def test_chi_cuadrado(df, variable, target='absentismo_elevado'):
    tabla = pd.crosstab(df[variable], df[target])

    if tabla.shape[0] < 2 or tabla.shape[1] < 2:
        return None

    chi2, p_value, dof, expected = stats.chi2_contingency(tabla)

    return {
        'variable': variable,
        'test': 'Chi-cuadrado',
        'p_value': round(p_value, 4),
        'significativo_p_005': 'Sí' if p_value < 0.05 else 'No',
        'tabla': tabla
    }

variables_chi_tests = variables_binarias_tests + variables_grupos_tests

chi_tests = pd.DataFrame([
    {k: v for k, v in test_chi_cuadrado(df_empleados_tests, variable).items() if k != 'tabla'}
    for variable in variables_chi_tests
    if test_chi_cuadrado(df_empleados_tests, variable) is not None
])

chi_tests


**Lectura Chi-cuadrado:** el contraste con `absentismo_elevado` confirma señales en algunas categorías, especialmente `tramo_transporte` y `tramo_hit_target`. `Disciplinary_failure` puede aparecer asociado al agrupar por ID, pero se interpreta solo como indicador administrativo contextual, porque sus registros propios tienen 0 horas de absentismo.

In [ ]:
# Resumen ejecutivo de los tests sobre variables personales/laborales
resumen_tests = pd.DataFrame({
    'variable': [
        'Transportation_expense', 'tramo_transporte', 'tramo_hit_target',
        'Disciplinary_failure', 'tramo_antiguedad', 'Son', 'Social_smoker',
        'Age / tramo_edad', 'Distance_Residence_Work / tramo_distancia',
        'Education_name', 'Pet', 'BMI_calculated / tramo_imc', 'Work_load_Average_day'
    ],
    'resultado': [
        'Spearman significativo pero débil (rho ≈ 0,32)',
        'Kruskal y Chi-cuadrado significativos por tramos',
        'Kruskal y Chi-cuadrado significativos al categorizar Hit_target',
        'Significativo al agregar por ID, pero sus filas propias tienen 0 horas',
        'Kruskal significativo, Chi-cuadrado no concluyente',
        'Spearman débil significativo; por grupos no concluyente',
        'Mann-Whitney significativo, pero grupo fumador muy pequeño',
        'Sin evidencia clara',
        'Sin evidencia clara',
        'Sin evidencia clara',
        'Sin evidencia clara',
        'Sin evidencia clara',
        'Sin evidencia clara'
    ],
    'deduccion': [
        'Puede orientar una revisión de movilidad/coste de desplazamiento, no prueba causa.',
        'Los tramos medio-alto/alto concentran más absentismo elevado; investigar movilidad y horarios.',
        'La relación no es lineal; útil como señal exploratoria, no como explicación de desempeño.',
        'No explica horas de ausencia; solo indica que algunos IDs con nota también acumulan horas en otros registros.',
        'Conviene revisar etapa laboral y acompañamiento, especialmente grupos pequeños.',
        'Puede orientar preguntas de conciliación, sin afirmar causalidad.',
        'No usar como conclusión fuerte por tamaño muestral reducido.',
        'No priorizar como eje explicativo del absentismo.',
        'No priorizar como eje explicativo; puede revisarse como movilidad junto a transporte.',
        'No priorizar como eje explicativo por desbalance de grupos.',
        'No priorizar como eje explicativo.',
        'No priorizar como eje explicativo y tratar IMC con cautela ética.',
        'No parece separar absentismo por sí sola.'
    ]
})

resumen_tests


Además del p-valor, revisamos el tamaño y el comportamiento de cada segmento. Esta tabla ayuda a leer los tests con criterio de negocio: no basta con que algo sea significativo; también debe ser interpretable, suficientemente estable y útil para una política interna.

In [ ]:
variables_cierre = [
    'Tiene_hijos', 'Tiene_mascotas', 'Educacion_binaria',
    'tramo_edad', 'tramo_antiguedad', 'tramo_distancia',
    'tramo_transporte', 'tramo_hit_target', 'tramo_imc',
    'Social_drinker', 'Social_smoker', 'Disciplinary_failure'
]

resumen_segmentos = []
for variable in variables_cierre:
    tabla_variable = (
        df_empleados_tests
        .groupby(variable, observed=False)
        .agg(
            ids=('ID', 'nunique'),
            mediana_horas=('horas_totales', 'median'),
            media_horas=('horas_totales', 'mean'),
            pct_absentismo_elevado=('absentismo_elevado', 'mean')
        )
        .reset_index()
    )
    tabla_variable['variable'] = variable
    tabla_variable['pct_absentismo_elevado'] = (tabla_variable['pct_absentismo_elevado'] * 100).round(1)
    tabla_variable['media_horas'] = tabla_variable['media_horas'].round(1)
    resumen_segmentos.append(tabla_variable)

resumen_segmentos = pd.concat(resumen_segmentos, ignore_index=True)
resumen_segmentos = resumen_segmentos[['variable'] + [col for col in resumen_segmentos.columns if col != 'variable']]
resumen_segmentos

### Conclusiones de los tests

Los contrastes permiten cerrar el estudio de variables personales y laborales antes de pasar a la parte operativa:

- **No aparece una variable individual fuerte** que explique por sí sola el absentismo elevado.
- La señal más consistente es **Transportation_expense / tramo_transporte**: apunta a revisar movilidad, coste de desplazamiento y posibles ajustes horarios, pero no prueba causalidad.
- **Hit_target categorizado** muestra diferencias por tramos, aunque la relación no es lineal. Conviene leerlo como contexto de desempeño agregado, no como causa directa.
- **Disciplinary_failure** no debe interpretarse como causa ni como ausencia: cuando vale 1, las horas del registro son 0. Su asociación aparece solo en la vista agregada por ID, porque algunos empleados con nota disciplinaria también tienen otras ausencias registradas.
- **Antigüedad por tramos** aparece como señal exploratoria; puede orientar acompañamiento por etapa laboral.
- **Hijos** y **Social_smoker** muestran señales débiles o sensibles al tamaño del grupo, por lo que no deben convertirse en políticas directas.
- **Edad, distancia, educación, mascotas, IMC y carga media** no muestran evidencia clara para explicar el absentismo elevado.

Deducción principal: sí se revisaron las variables personales y laborales propuestas, tanto originales como categorizadas. El resultado no sostiene una política basada en “perfiles individuales”. Por eso el análisis continúa hacia impacto operativo: motivos, momentos, concentración y segmentos agregados para decidir dónde intervenir sin estigmatizar.


## 7. Cambio de enfoque: del perfil individual al impacto operativo

Como no aparece una variable individual fuerte, la respuesta debe centrarse en impacto operativo:

- qué motivos acumulan más horas;
- cuándo se concentra la carga;
- si pocas personas/IDs concentran muchas horas;
- qué segmentos agregados conviene investigar sin etiquetar personas.


## 8. Motivos de ausencia: frecuencia vs impacto


In [ ]:
absentismo_motivo = df_RRHH.groupby(['Reason_absence', 'Reason_absence_name'], as_index=False).agg(
    registros=('ID', 'size'),
    ids_observados=('ID', 'nunique'),
    horas_totales=('Absenteeism_hours', 'sum'),
    media_horas=('Absenteeism_hours', 'mean'),
    mediana_horas=('Absenteeism_hours', 'median')
).sort_values('horas_totales', ascending=False)

# Para presentación usamos top 10 por impacto acumulado en horas.
top_motivos = absentismo_motivo.head(10).copy()
top_motivos


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_alpha(0)
ax.set_facecolor('none')

# Escalamos el tamaño de las burbujas con numpy para que todas sean visibles.
size_min = 550
size_max = 1900
media = top_motivos['media_horas']
if media.max() == media.min():
    bubble_sizes = np.repeat((size_min + size_max) / 2, len(top_motivos))
else:
    bubble_sizes = np.interp(media, (media.min(), media.max()), (size_min, size_max))

# Un color legible para todos los motivos y un destacado para el motivo de mayor impacto.
colors = [COLOR_VERDE_OSCURO if reason == 13 else COLOR_VERDE_MEDIO for reason in top_motivos['Reason_absence']]

ax.scatter(
    top_motivos['registros'],
    top_motivos['horas_totales'],
    s=bubble_sizes,
    c=colors,
    edgecolors='white',
    linewidths=2.2,
    alpha=0.92
)

for _, fila in top_motivos.iterrows():
    ax.text(
        fila['registros'], fila['horas_totales'], int(fila['Reason_absence']),
        ha='center', va='center', color='white', fontweight='bold', fontsize=10
    )

ax.set_title('Motivos principales: frecuencia vs horas acumuladas', fontsize=14, fontweight='bold', color=COLOR_TEXTO)
ax.set_xlabel('Registros')
ax.set_ylabel('Horas totales')
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
import plotly.express as px

fig = px.scatter(
    top_motivos,
    x='registros',
    y='horas_totales',
    size='media_horas',
    color='Reason_absence_name',
    text='Reason_absence',
    hover_data={
        'Reason_absence_name': True,
        'registros': True,
        'horas_totales': True,
        'media_horas': ':.2f',
        'mediana_horas': True
    },
    title='Motivos principales: frecuencia vs horas acumuladas'
)

fig.show()

**Lectura:** este gráfico separa recurrencia e impacto. Los motivos con más horas acumuladas orientan mejor las políticas de prevención que una lectura basada solo en frecuencia.


## 9. Días y meses con mayor carga


In [ ]:
absentismo_mes = df_RRHH.groupby(['Month_absence', 'Month_absence_name'], as_index=False).agg(
    registros=('ID', 'size'),
    horas_totales=('Absenteeism_hours', 'sum')
).sort_values('Month_absence')

absentismo_dia = df_RRHH.groupby(['Day_week', 'Day_week_name'], as_index=False).agg(
    registros=('ID', 'size'),
    horas_totales=('Absenteeism_hours', 'sum')
).sort_values('Day_week')


In [ ]:
data = absentismo_mes.copy()
total_horas = data['horas_totales'].sum()
data['pct_horas'] = data['horas_totales'] / total_horas * 100

norm = plt.Normalize(data['horas_totales'].min(), data['horas_totales'].max())
colors = plt.cm.Greens(norm(data['horas_totales']))

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_alpha(0)
ax.set_facecolor('none')

sns.barplot(
    data=data,
    x='Month_absence_name',
    y='horas_totales',
    hue='Month_absence_name',
    palette=list(colors),
    legend=False,
    ax=ax
)

ax.set_title('Horas de absentismo por mes', fontsize=14, fontweight='bold', color=COLOR_TEXTO)
ax.set_xlabel('Mes')
ax.set_ylabel('Horas de absentismo')
ax.tick_params(axis='x', rotation=45)

for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', padding=3, fontsize=9)

for i, fila in data.reset_index(drop=True).iterrows():
    color_texto = 'white' if fila['horas_totales'] > data['horas_totales'].median() else COLOR_TEXTO
    ax.text(i, fila['horas_totales'] / 2, f"{fila['pct_horas']:.1f}%", ha='center', va='center', color=color_texto, fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
total_horas = df_RRHH['Absenteeism_hours'].sum()
absentismo_dia['pct_horas'] = absentismo_dia['horas_totales'] / total_horas * 100

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_alpha(0)
ax.set_facecolor('none')

sns.barplot(data=absentismo_dia, x='Day_week_name', y='horas_totales', color=COLOR_OCRE, ax=ax)
ax.set_title('Horas por día de la semana', fontsize=13, fontweight='bold', color=COLOR_TEXTO)
ax.set_xlabel('Día')
ax.set_ylabel('Horas totales')
ax.tick_params(axis='x', rotation=25)

for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', padding=3, fontsize=9)

for i, fila in absentismo_dia.reset_index(drop=True).iterrows():
    ax.text(i, fila['horas_totales'] / 2, f"{fila['pct_horas']:.1f}%", ha='center', va='center', color='white', fontweight='bold')

plt.tight_layout()
plt.show()


**Lectura:** meses y días con mayor carga sirven para planificar cobertura, turnos o sustituciones. La limitación es que el dataset no incluye año, por lo que no permite una evolución temporal precisa.


## 10. Concentración por ID


In [ ]:
absentismo_id = df_RRHH.groupby('ID', as_index=False).agg(
    registros=('ID', 'size'),
    horas_totales=('Absenteeism_hours', 'sum')
).sort_values('horas_totales', ascending=False)

top_ids = absentismo_id.head(10)

top_10_share = top_ids['horas_totales'].sum() / df_RRHH['Absenteeism_hours'].sum() * 100
top_5_share = absentismo_id.head(5)['horas_totales'].sum() / df_RRHH['Absenteeism_hours'].sum() * 100

print('Top 10 IDs concentran:', round(top_10_share, 1), '% de las horas')
print('Top 5 IDs concentran:', round(top_5_share, 1), '% de las horas')

top_ids


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_alpha(0)
ax.set_facecolor('none')
sns.barplot(data=top_ids, x='ID', y='horas_totales', color=COLOR_TEAL, order=top_ids['ID'], ax=ax)
ax.set_title('Top 10 IDs por horas registradas', fontsize=14, fontweight='bold')
ax.set_xlabel('ID anonimizado')
ax.set_ylabel('Horas totales')

for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', padding=3, fontsize=9)

plt.tight_layout()
plt.show()


**Lectura:** la concentración por ID no debe usarse para señalar personas. Sirve para detectar si el impacto requiere seguimiento confidencial con más contexto: puesto, centro, turno, contrato y situación laboral.


## 10.1. Pareto de horas por ID

El Pareto ayuda a comprobar si una parte pequeña de IDs concentra una parte grande de las horas de absentismo.

No se usa para señalar personas. Se usa para decidir si RRHH necesita una vía de seguimiento confidencial con más contexto.


In [ ]:
pareto_id = absentismo_id.copy()
pareto_id['pct_horas'] = pareto_id['horas_totales'] / pareto_id['horas_totales'].sum() * 100
pareto_id['pct_acumulado'] = pareto_id['pct_horas'].cumsum()
pareto_id['orden'] = range(1, len(pareto_id) + 1)

ids_hasta_80 = pareto_id[pareto_id['pct_acumulado'] <= 80].shape[0] + 1
pct_top_10 = pareto_id.head(10)['horas_totales'].sum() / pareto_id['horas_totales'].sum() * 100

print('IDs necesarios para alcanzar aproximadamente el 80% de las horas:', ids_hasta_80)
print('Top 10 IDs concentran:', round(pct_top_10, 1), '% de las horas')

pareto_id.head(20)


In [ ]:
pareto_top = pareto_id.head(20).copy()

fig, ax1 = plt.subplots(figsize=(12, 5))
fig.patch.set_alpha(0)
ax1.set_facecolor('none')

sns.barplot(
    data=pareto_top,
    x='orden',
    y='horas_totales',
    color=COLOR_TEAL,
    ax=ax1
)

ax1.set_title('Pareto de horas de absentismo por ID', fontsize=14, fontweight='bold')
ax1.set_xlabel('ID anonimizado')
ax1.set_ylabel('Horas totales')
ax1.set_xticks(range(len(pareto_top)))
ax1.set_xticklabels(pareto_top['ID'].astype(str), rotation=0)

for container in ax1.containers:
    ax1.bar_label(container, fmt='%.0f', padding=3, fontsize=8)

ax2 = ax1.twinx()
ax2.set_facecolor('none')
ax2.plot(
    range(len(pareto_top)),
    pareto_top['pct_acumulado'],
    color='black',
    marker='o'
)
ax2.axhline(80, color='gray', linestyle='--', linewidth=1)
ax2.set_ylabel('% acumulado de horas')
ax2.set_ylim(0, 100)

plt.tight_layout()
plt.show()


**Lectura:** el Pareto confirma una concentración importante: los **10 IDs con más horas acumulan cerca del 65,4%** del total, y hacen falta aproximadamente **16 IDs** para alcanzar el 80% de las horas.

Esto refuerza la idea de impacto operativo concentrado. La política no debe ser exponer IDs, sino abrir una revisión confidencial con datos de puesto, centro, turno, contrato y causa de ausencia.


## 11. Segmentos agregados para investigar

Como alternativa a una lectura individual, comparamos segmentos con el porcentaje de IDs en absentismo elevado.


In [ ]:
df_empleados['tramo_distancia'] = pd.cut(
    df_empleados['Distance_Residence_Work'],
    bins=[0, 10, 20, 30, 100],
    labels=['<=10 km', '11-20 km', '21-30 km', '>30 km']
)

df_empleados['tramo_antiguedad'] = pd.cut(
    df_empleados['Service_time'],
    bins=[0, 5, 10, 15, 40],
    labels=['<=5 años', '6-10 años', '11-15 años', '>15 años']
)

def comparar_segmentos(df, variable):
    tabla = df.groupby(variable, observed=False).agg(
        ids_observados=('ID', 'size'),
        horas_totales=('horas_totales', 'sum'),
        media_horas=('horas_totales', 'mean'),
        mediana_horas=('horas_totales', 'median'),
        ids_absentismo_elevado=('absentismo_elevado', 'sum')
    ).reset_index()

    tabla['pct_absentismo_elevado'] = (
        tabla['ids_absentismo_elevado'] / tabla['ids_observados'] * 100
    ).round(1)

    return tabla.sort_values('pct_absentismo_elevado', ascending=False)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.patch.set_alpha(0)
for ax in axes:
    ax.set_facecolor('none')

sns.barplot(data=comparar_segmentos(df_empleados, 'tramo_distancia'), x='tramo_distancia', y='pct_absentismo_elevado', ax=axes[0], color=COLOR_TEAL)

axes[0].set_xlabel('Distancia')
axes[0].set_ylabel('% IDs con absentismo elevado')
axes[0].tick_params(axis='x', rotation=30)

sns.barplot(data=comparar_segmentos(df_empleados, 'tramo_antiguedad'), x='tramo_antiguedad', y='pct_absentismo_elevado', ax=axes[1], color=COLOR_TEAL)

axes[1].set_xlabel('Antigüedad')
axes[1].set_ylabel('')
axes[1].tick_params(axis='x', rotation=30)

sns.barplot(data=comparar_segmentos(df_empleados, 'Son'), x='Son', y='pct_absentismo_elevado', ax=axes[2], color=COLOR_TEAL)

axes[2].set_xlabel('Hijos')
axes[2].set_ylabel('')

plt.tight_layout()
plt.show()


**Lectura:** estos segmentos no son perfiles de riesgo. Son señales para investigar conciliación, movilidad, etapa laboral y organización del trabajo con más datos.


## 12. Respuesta final a la pregunta de negocio

No aparece una variable personal o laboral que explique de forma fuerte el absentismo elevado. Esto coincide con el análisis de perfil/desempeño: las relaciones lineales con absentismo son débiles.

Por tanto, la respuesta no debe centrarse en señalar perfiles, sino en actuar sobre el impacto operativo:

- motivos que acumulan más horas;
- días y meses de mayor carga;
- concentración de horas en pocos IDs;
- segmentos agregados donde conviene investigar políticas internas.


## 13. Propuestas de políticas internas


In [ ]:
tabla_politicas = pd.DataFrame({
    'hallazgo': [
        'Motivos musculoesqueléticos concentran muchas horas',
        'Algunos días y meses concentran más carga registrada',
        'Pocos IDs acumulan una parte alta de las horas',
        'Hijos, distancia o transporte muestran señales exploratorias',
        'Correlaciones personales/laborales débiles'
    ],
    'interpretacion': [
        'El impacto parece más relacionado con salud laboral y condiciones físicas que con un perfil individual claro.',
        'El absentismo tiene impacto operativo desigual en el calendario disponible.',
        'La concentración requiere análisis confidencial y contextual, no exposición pública de personas.',
        'Pueden orientar preguntas sobre conciliación, movilidad u horarios, sin asumir causalidad.',
        'No conviene construir políticas basadas en perfiles individuales o modelos lineales simples.'
    ],
    'posible_politica_interna': [
        'Prevención ergonómica, revisión de cargas físicas y seguimiento de salud laboral.',
        'Planificación de cobertura, turnos y sustituciones en momentos de mayor presión.',
        'Seguimiento confidencial con información de puesto, centro, turno y contrato.',
        'Revisar medidas de conciliación, flexibilidad, transporte u organización horaria.',
        'Usar segmentos agregados e impacto operativo como base para priorizar decisiones.'
    ]
})

tabla_politicas


### Cierre

La conclusión principal es que el absentismo no se resuelve señalando perfiles individuales. La intervención más útil para RRHH debe combinar prevención, planificación operativa y análisis agregado de segmentos a investigar.
